<a href="https://colab.research.google.com/github/10dimensions/gnc-toolbox/blob/main/sensor_error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np

In [3]:
def simulate_gyro_errors():
    print("--- GNC Sensor Error Simulation ---\n")

    # =========================================================================
    # 1. Simulation Setup
    # =========================================================================
    dt = 0.01          # Time step (100 Hz update rate)
    t_total = 3600.0   # 1 hour of operation (3600 seconds)
    t = np.arange(0, t_total, dt)

    # True spacecraft rotation rate (deg/s)
    true_rate = 0.5

    # =========================================================================
    # 2. Define Sensor Error Parameters (Typical Space-Grade FOG)
    # =========================================================================
    # A) White Noise (Rate Noise)
    # High-frequency jitter. Standard deviation per sample.
    white_noise_std = 0.005  # deg/s

    # B) Bias Instability (Random Walk)
    # Low-frequency drift. How much the bias changes per second.
    # (Modeled as the integral of a white noise process)
    bias_walk_std = 0.0002   # deg/s / sqrt(second)

    # =========================================================================
    # 3. Generate the Errors
    # =========================================================================
    np.random.seed(99) # For reproducibility

    # White noise: independent random samples at each time step
    eta_v = np.random.normal(0, white_noise_std, len(t))

    # Bias Instability: modeled as a Random Walk
    # The driving noise for the bias (scaled by sqrt(dt) for discrete integration)
    eta_u = np.random.normal(0, bias_walk_std * np.sqrt(dt), len(t))
    bias = np.cumsum(eta_u) # Integration = cumulative sum

    # =========================================================================
    # 4. Construct the Measured Signal
    # =========================================================================
    # Equation: measured = true + bias + white_noise
    measured_rate = true_rate + bias + eta_v

    # Calculate total error for analysis
    error = measured_rate - true_rate

    # =========================================================================
    # 5. Analyze and Print Results
    # =========================================================================
    print(f"Simulating {t_total} seconds of Gyro Data at {1/dt} Hz\n")
    print(f"{'Time (s)':<12} {'True Rate':<12} {'Measured Rate':<16} {'Total Error':<14} {'Bias Drift':<12}")
    print("-" * 70)

    # Checkpoints to show how the error evolves
    checkpoints = [1, 10, 60, 300, 1800, 3600]
    for t_check in checkpoints:
        idx = int(t_check / dt) - 1
        print(f"{t_check:<12} {true_rate:<12.4f} {measured_rate[idx]:<16.4f} {error[idx]:<14.4f} {bias[idx]:<12.4f}")

    print("\n--- Error Analysis ---")
    # Calculate RMS of the white noise vs the final bias drift
    rms_white = np.std(eta_v)
    final_bias_drift = np.abs(bias[-1])

    print(f"White Noise RMS (constant over time): {rms_white:.4f} deg/s")
    print(f"Final Bias Drift after 1 hour:        {final_bias_drift:.4f} deg/s")
    print(f"\n-> Notice how White Noise stays constant, but Bias Drift grows continuously!")
    print(f"-> If we integrate this rate to get angle, the bias drift will cause massive error.")

In [4]:
if __name__ == "__main__":
    simulate_gyro_errors()

--- GNC Sensor Error Simulation ---

Simulating 3600.0 seconds of Gyro Data at 100.0 Hz

Time (s)     True Rate    Measured Rate    Total Error    Bias Drift  
----------------------------------------------------------------------
1            0.5000       0.5073           0.0073         -0.0002     
10           0.5000       0.4996           -0.0004        0.0007      
60           0.5000       0.5111           0.0111         0.0068      
300          0.5000       0.5116           0.0116         0.0099      
1800         0.5000       0.4993           -0.0007        -0.0065     
3600         0.5000       0.5065           0.0065         0.0053      

--- Error Analysis ---
White Noise RMS (constant over time): 0.0050 deg/s
Final Bias Drift after 1 hour:        0.0053 deg/s

-> Notice how White Noise stays constant, but Bias Drift grows continuously!
-> If we integrate this rate to get angle, the bias drift will cause massive error.
